# Phase 2: Fine-Tune SmolVLA on LIBERO

Fine-tune the SmolVLA (450M params) model on LIBERO human demonstrations using the LeRobot framework.

**What we do here:**
1. Install LeRobot with SmolVLA support
2. Load LIBERO demo data (already in LeRobot format on HuggingFace)
3. Fine-tune SmolVLA from pretrained checkpoint
4. Fix post-training config (n_action_steps)
5. Upload trained model to HuggingFace Hub

**Runtime:** GPU with 16GB+ VRAM (A100 preferred, T4 works with smaller batch)

**Time:** ~2-4 hours for 20k steps on A100

## 1. Install LeRobot + SmolVLA

In [ ]:
%%bash
# Install LeRobot with SmolVLA support
if [ ! -d "lerobot" ]; then
    git clone https://github.com/huggingface/lerobot.git
fi
cd lerobot && pip install -q -e ".[smolvla]"

# Pin numpy for compatibility
pip install -q "numpy>=2.0,<2.1"

echo "LeRobot + SmolVLA installed"

In [ ]:
# Restart runtime after install
import os
os.kill(os.getpid(), 9)

## 2. Setup + Verify (run after restart)

In [ ]:
import os
import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import lerobot
print(f"LeRobot: {lerobot.__version__}")
print(f"numpy: {np.__version__}")

## 3. Configure Training

SmolVLA is pretrained on 10M frames from 487 LeRobot datasets. We fine-tune from this checkpoint.

In [ ]:
# Training configuration
SUITE = "libero_spatial"  # Change for other suites
# HuggingFaceVLA/libero is the public v3.0 format dataset compatible with LeRobot v0.4+
# (yifengzhu-hf datasets are gated and use v2.0 format which is incompatible)
DATASET_REPO = "HuggingFaceVLA/libero"
MODEL = "lerobot/smolvla_base"  # Pretrained SmolVLA
OUTPUT_DIR = f"outputs/smolvla_{SUITE}"

# HuggingFace username for policy.repo_id (required by LeRobot)
HF_USERNAME = "your-username"  # Change this to your HF username

# Hyperparameters
BATCH_SIZE = 64    # Reduce to 32 or 16 if OOM on T4
TOTAL_STEPS = 20000
SAVE_FREQ = 5000
EVAL_FREQ = 2000
SEED = 42

# W&B logging (optional)
USE_WANDB = False  # Set True and configure wandb login
WANDB_PROJECT = "smolvla-libero"

print(f"Suite: {SUITE}")
print(f"Dataset: {DATASET_REPO}")
print(f"Model: {MODEL}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps: {TOTAL_STEPS}")
print(f"Output: {OUTPUT_DIR}")

## 4. Launch Training

Uses `lerobot-train` CLI (LeRobot v0.4+). On A100: ~2-4 hours for 20k steps.

**Note:** The dataset (~8GB) will download on first run. This takes 10-15 minutes.
Do NOT interrupt the cell during download.

In [ ]:
# Build training command using lerobot-train CLI (not python -m lerobot.scripts.train)
# String concatenation avoids f-string backslash issues in Colab
cmd = (
    "lerobot-train"
    " --policy.path=" + MODEL +
    " --policy.repo_id=" + HF_USERNAME + "/smolvla-" + SUITE +
    " --dataset.repo_id=" + DATASET_REPO +
    " --batch_size=" + str(BATCH_SIZE) +
    " --steps=" + str(TOTAL_STEPS) +
    " --output_dir=" + OUTPUT_DIR +
    " --save_freq=" + str(SAVE_FREQ) +
    " --eval_freq=" + str(EVAL_FREQ) +
    " --seed=" + str(SEED) +
    " --policy.device=cuda"
)

if USE_WANDB:
    cmd += " --wandb.enable=true --wandb.project=" + WANDB_PROJECT

print("Training command:")
print(cmd)
print("\nStarting training...")
!{cmd}

## 5. Post-Training: Fix n_action_steps

**CRITICAL**: SmolVLA uses action chunking (50 actions per inference). The training script
sets `n_action_steps=1`, but inference needs `n_action_steps=50`. Without this fix,
inference is 50x slower.

In [ ]:
import json
from pathlib import Path

# Redefine OUTPUT_DIR in case runtime was restarted
SUITE = "libero_spatial"
OUTPUT_DIR = f"outputs/smolvla_{SUITE}"

# Find config.json in output directory
output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    print(f"ERROR: Directory does not exist: {output_dir}")
    print("Training must complete successfully before running this cell.")
else:
    config_files = list(output_dir.rglob("config.json"))

    for config_path in config_files:
        with open(config_path) as f:
            cfg = json.load(f)

        old_val = cfg.get("n_action_steps", "not set")
        cfg["n_action_steps"] = 50

        with open(config_path, "w") as f:
            json.dump(cfg, f, indent=2)

        print(f"Fixed {config_path}: n_action_steps {old_val} -> 50")

    if not config_files:
        print(f"WARNING: No config.json found in {output_dir}")

## 6. Upload to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and configure to upload
# from huggingface_hub import HfApi, login
# login()  # Will prompt for HF token

# HF_USERNAME = "your-username"
# REPO_ID = f"{HF_USERNAME}/smolvla-{SUITE}"

# api = HfApi()
# api.create_repo(REPO_ID, exist_ok=True)
# api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID)
# print(f"Uploaded to https://huggingface.co/{REPO_ID}")

## 7. Save Checkpoint to Google Drive

In [ ]:
# Uncomment to save
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {OUTPUT_DIR} /content/drive/MyDrive/AUTOLAB/checkpoints/
# print("Saved to Google Drive")

## Phase 2 Complete!

**What we did:**
- Fine-tuned SmolVLA (450M) on LIBERO human demonstrations
- Fixed n_action_steps for proper inference speed
- Saved/uploaded trained checkpoint

**Next:** Open `03_evaluate.ipynb` to evaluate on standard LIBERO and LIBERO-PRO.